###

In [2]:
import os
from dotenv import load_dotenv
from graphdatascience import GraphDataScience
import pandas as pd

In [ ]:
#Storing Neo4j connection details in variables
URI = "neo4j://127.0.0.1:7687"
USER = "neo4j"
PASSWORD = ""
DB_NAME = ""

In [4]:
gds = GraphDataScience(
    URI,
    auth=(USER, PASSWORD),
    database=DB_NAME
)
print("Connected to Neo4j GDS server version:", gds.version())

Connected to Neo4j GDS server version: 2.25.0


In [5]:
dense_wcc_id = 2 

In [6]:
def drop_if_exists(graph_name: str):
    graphs = gds.graph.list()
    if "graphName" in graphs.columns and graph_name in graphs["graphName"].values:
        gds.graph.drop(graph_name)

In [10]:
if gds.graph.exists("foodweb_directed").exists:
    gds.graph.drop("foodweb_directed")

In [11]:
if gds.graph.exists("foodweb_undirected").exists:
    gds.graph.drop("foodweb_undirected")

In [29]:
G, proj_result = gds.graph.project(
    "foodweb_directed",
    {
        "Species": {
            "properties": []
        }
    },
    {
        "eaten_by": {
            "orientation": "NATURAL",   
            "properties": []
        }
    }
)


In [12]:
G_und, proj_result = gds.graph.project(
    "foodweb_undirected",
    {
        "Species": {
            "properties": []
        }
    },
    {
        "eaten_by": {
            "orientation": "UNDIRECTED",   
            "properties": []
        }
    }
)

In [13]:
dense = gds.wcc.write(
    G_und, 
    writeProperty="wccId"
)
print("WCC IDs written for predator to prey graph as 'wccId' property.\n")

WCC IDs written for predator to prey graph as 'wccId' property.



In [14]:
# 1) Dense: predator -> prey  (reverse edges for "who points to prey" view)
drop_if_exists("dense_pred_to_prey")
G_dense_pred_to_prey, _ = gds.graph.project.cypher(
    "dense_pred_to_prey",
    f"""
    MATCH (s:Species)
    WHERE s.wccId = {dense_wcc_id}
    RETURN id(s) AS id
    """,
    f"""
    MATCH (prey:Species)-[:eaten_by]->(pred:Species)
    WHERE prey.wccId = {dense_wcc_id} AND pred.wccId = {dense_wcc_id}
    RETURN id(pred) AS source, id(prey) AS target
    """
)
print("Dense projections created: - dense_pred_to_prey")

Dense projections created: - dense_pred_to_prey


In [15]:
# 2) Dense: prey -> predator  (:eaten_by means prey -> predator)
drop_if_exists("dense_prey_to_pred")
G_dense_prey_to_pred, _ = gds.graph.project.cypher(
    "dense_prey_to_pred",
    f"""
    MATCH (s:Species)
    WHERE s.wccId = {dense_wcc_id}
    RETURN id(s) AS id
    """,
    f"""
    MATCH (prey:Species)-[:eaten_by]->(pred:Species)
    WHERE prey.wccId = {dense_wcc_id} AND pred.wccId = {dense_wcc_id}
    RETURN id(prey) AS source, id(pred) AS target
    """
)
print("Dense projections created: - dense_prey_to_pred")

Dense projections created: - dense_prey_to_pred


In [ ]:
# 3) Dense UNDIRECTED (strict): add reverse edges so it behaves undirected
drop_if_exists("dense_undirected")
G_dense_undirected, _ = gds.graph.project.cypher(
    "dense_undirected",
    f"""
    MATCH (s:Species)
    WHERE s.wccId = {dense_wcc_id}
    RETURN id(s) AS id
    """,
    f"""
    MATCH (a:Species)-[:eaten_by]->(b:Species)
    WHERE a.wccId = {dense_wcc_id} AND b.wccId = {dense_wcc_id}
    RETURN id(a) AS source, id(b) AS target
    UNION
    MATCH (a:Species)-[:eaten_by]->(b:Species)
    WHERE a.wccId = {dense_wcc_id} AND b.wccId = {dense_wcc_id}
    RETURN id(b) AS source, id(a) AS target
    """
)
print("Dense projections created: - dense_undirected")

Dense projections created: - dense_undirected


In [45]:
drop_if_exists("dense_undirected")
G_und_dense, proj_result = gds.graph.project(
    "dense_undirected",
    {
        "Species": {
            "properties": ["wccId"],
            "where": f"wccId = {dense_wcc_id}"
        }
    },
    {
        "eaten_by": {
            "orientation": "UNDIRECTED",   
            "properties": []
        }
    }
)

ClientError: {neo4j_code: Neo.ClientError.Procedure.ProcedureCallFailed} {message: Failed to invoke procedure `gds.graph.project`: Caused by: java.lang.IllegalArgumentException: Unexpected configuration key: where} {gql_status: 52N37} {gql_status_description: error: procedure exception - procedure execution error. Execution of the procedure gds.graph.project() failed.}

In [46]:
# 3) Dense UNDIRECTED (strict): add reverse edges so it behaves undirected
drop_if_exists("dense_undirected")
G_dense_undirected, _ = gds.graph.project(
    "dense_undirected",
    f"""
    MATCH (s:Species)
    WHERE s.wccId = {dense_wcc_id}
    RETURN id(s) AS id
    """,
   {
        "eaten_by": {
            "orientation": "UNDIRECTED",   
            "properties": []
        }
    }
)
print("Dense projections created: - dense_undirected")

ClientError: {neo4j_code: Neo.ClientError.Procedure.ProcedureCallFailed} {message: Failed to invoke procedure `gds.graph.project`: Caused by: java.lang.IllegalArgumentException: Invalid node projection, one or more labels not found: '
    MATCH (s:Species)
    WHERE s.wccId = 2
    RETURN id(s) AS id
    '} {gql_status: 52N37} {gql_status_description: error: procedure exception - procedure execution error. Execution of the procedure gds.graph.project() failed.}

### Finding primary consumers using reverse projection

In [18]:
query = """// Step 1: Identify producers
CALL gds.degree.stream(
  'dense_pred_to_prey',
  { orientation: 'NATURAL' }
)
YIELD nodeId, score AS indegree
WHERE indegree = 0
WITH gds.util.asNode(nodeId) AS producer
WHERE producer.taxon_kingdom IN ["Plantae", "Chromista"]

// Step 2: Find primary consumers
MATCH (producer)-[:eaten_by]->(primary:Species)

// Step 3: Return
RETURN DISTINCT primary.scientific_name AS scientific_name,
       primary.common_name AS common_name,
       primary.taxon_class AS class,
       primary.taxon_phylum AS phylum
ORDER BY scientific_name;"""

result = gds.run_cypher(query)
print(result)

               scientific_name                  common_name      class  \
0              Aceria lantanae      Lantana Flower Gallmite  Arachnida   
1             Aceria theospyri  persimmon leaf blister gall  Arachnida   
2           Adejeania vexatrix                         None    Insecta   
3          Aerophilus nigripes                         None    Insecta   
4                  Afrogegenes                      Dodgers    Insecta   
..                         ...                          ...        ...   
216          Zonocerus elegans                         None    Insecta   
217  Zonocerus elegans elegans                         None    Insecta   
218           Zosterops virens         Green Cape White-eye       Aves   
219    Zosterops virens virens         Green Cape White-eye       Aves   
220       Zygaena filipendulae              Six-spot Burnet    Insecta   

         phylum  
0    Arthropoda  
1    Arthropoda  
2    Arthropoda  
3    Arthropoda  
4    Arthropoda  
.. 

### Finding producers

In [20]:
query = """CALL gds.degree.stream(
  'dense_prey_to_pred',
{ orientation: 'REVERSE' })
YIELD nodeId, score AS indegree
WHERE indegree = 0 
WITH gds.util.asNode(nodeId) AS producer
WHERE producer.taxon_kingdom in ["Plantae", "Chromista"]
RETURN producer
ORDER BY producer.name;"""
result = gds.run_cypher(query)
print(result)

                                              producer
0    (iconic_taxon_name, taxon_kingdom, taxon_subph...
1    (iconic_taxon_name, taxon_subfamily, taxon_kin...
2    (iconic_taxon_name, taxon_subfamily, taxon_kin...
3    (iconic_taxon_name, taxon_subfamily, taxon_kin...
4    (iconic_taxon_name, taxon_subfamily, taxon_kin...
..                                                 ...
177  (iconic_taxon_name, taxon_subfamily, taxon_kin...
178  (iconic_taxon_name, taxon_subfamily, taxon_kin...
179  (iconic_taxon_name, taxon_subfamily, taxon_kin...
180  (iconic_taxon_name, taxon_subfamily, taxon_kin...
181  (iconic_taxon_name, taxon_kingdom, taxon_subph...

[182 rows x 1 columns]


In [21]:
query = """CALL gds.degree.stream(
  'dense_pred_to_prey',
{ orientation: 'NATURAL' })
YIELD nodeId, score AS indegree
WHERE indegree = 0 
WITH gds.util.asNode(nodeId) AS producer
WHERE producer.taxon_kingdom in ["Plantae", "Chromista"]
RETURN producer
ORDER BY producer.name;"""
result = gds.run_cypher(query)
print(result)

                                              producer
0    (iconic_taxon_name, taxon_kingdom, taxon_subph...
1    (iconic_taxon_name, taxon_subfamily, taxon_kin...
2    (iconic_taxon_name, taxon_subfamily, taxon_kin...
3    (iconic_taxon_name, taxon_subfamily, taxon_kin...
4    (iconic_taxon_name, taxon_subfamily, taxon_kin...
..                                                 ...
177  (iconic_taxon_name, taxon_subfamily, taxon_kin...
178  (iconic_taxon_name, taxon_subfamily, taxon_kin...
179  (iconic_taxon_name, taxon_subfamily, taxon_kin...
180  (iconic_taxon_name, taxon_subfamily, taxon_kin...
181  (iconic_taxon_name, taxon_kingdom, taxon_subph...

[182 rows x 1 columns]


### Finding primary consumers using original projection

In [19]:
query = """CALL gds.degree.stream(
  'dense_prey_to_pred',
{ orientation: 'REVERSE' })
YIELD nodeId, score AS indegree
WHERE indegree = 0 
WITH gds.util.asNode(nodeId) AS producer
WHERE producer.taxon_kingdom in ["Plantae", "Chromista"]

// Step 2: Find primary consumers
MATCH (producer)-[:eaten_by]->(primary:Species)

RETURN primary.scientific_name as scientific_name,
        primary.common_name as common_name,
        primary.taxon_class as class,
        primary.taxon_phylum as phylum
ORDER BY producer.scientific_name;"""

result = gds.run_cypher(query)
print(result)

                           scientific_name  \
0                  Tamiasciurus hudsonicus   
1                        Trichodes ornatus   
2                         Vanessa atalanta   
3                   Strangalia luteicornis   
4                      Toxomerus geminatus   
..                                     ...   
431                Coniodictyum chevalieri   
432                        Giraffa giraffa   
433                Giraffa giraffa giraffa   
434               Tragelaphus strepsiceros   
435  Tragelaphus strepsiceros strepsiceros   

                              common_name              class         phylum  
0                   American Red Squirrel           Mammalia       Chordata  
1                 Ornate Checkered Beetle            Insecta     Arthropoda  
2                             Red Admiral            Insecta     Arthropoda  
3    Yellow-horned Flower Longhorn Beetle            Insecta     Arthropoda  
4                    Eastern Calligrapher            Inse

### Finding Secondary Consumers

In [22]:
query = """CALL gds.degree.stream(
  'dense_prey_to_pred',
{ orientation: 'REVERSE' })
YIELD nodeId, score AS indegree
WHERE indegree = 0 
WITH gds.util.asNode(nodeId) AS producer
WHERE producer.taxon_kingdom in ["Plantae", "Chromista"]

// Step 2: Find primary consumers
MATCH (producer)-[:eaten_by]->(primary:Species)

//Step 3: Find Secondary Consumers
MATCH (primary)-[:eaten_by]->(secondary:Species)

RETURN DISTINCT secondary.scientific_name as scientific_name,
        secondary.common_name as common_name,
        secondary.taxon_class as class,
        secondary.taxon_phylum as phylum
ORDER BY secondary.scientific_name;"""

result = gds.run_cypher(query)
print(result)

                scientific_name                   common_name  \
0             Aerospiza tachiro               African Goshawk   
1    Alligator mississippiensis            American Alligator   
2            Amblyomma hebraeum       South African Bont Tick   
3         Andrenosoma hesperium     Golden-horned Chiselmouth   
4        Apiomerus californicus       California Bee Assassin   
5                Apis mellifera             Western Honey Bee   
6           Argiope trifasciata          Banded Garden Spider   
7            Brunneria borealis         Northern Grass Mantis   
8             Buteo jamaicensis               Red-tailed Hawk   
9                 Canis latrans                        Coyote   
10               Cathartes aura                Turkey Vulture   
11          Chromolaena odorata                     Siam weed   
12             Coragyps atratus                 Black Vulture   
13         Crocodylus niloticus                Nile Crocodile   
14  Crocodylus niloticus 

### Finding Tertiary Consumers

In [23]:
query = """CALL gds.degree.stream(
  'dense_prey_to_pred',
{ orientation: 'REVERSE' })
YIELD nodeId, score AS indegree
WHERE indegree = 0 
WITH gds.util.asNode(nodeId) AS producer
WHERE producer.taxon_kingdom in ["Plantae", "Chromista"]

// Step 2: Find primary consumers
MATCH (producer)-[:eaten_by]->(primary:Species)

//Step 3: Find Secondary Consumers
MATCH (primary)-[:eaten_by]->(secondary:Species)

//Step 4: Find Tertiary Consumers
MATCH (secondary)-[:eaten_by]->(tertiary:Species)

RETURN DISTINCT tertiary.scientific_name as scientific_name,
        tertiary.common_name as common_name,
        tertiary.taxon_class as class,
        tertiary.taxon_phylum as phylum
ORDER BY tertiary.scientific_name;"""

result = gds.run_cypher(query)
print(result)

                scientific_name                          common_name  \
0                   Afrogegenes                              Dodgers   
1         Andrenosoma hesperium            Golden-horned Chiselmouth   
2        Apiomerus californicus              California Bee Assassin   
3                Apis mellifera                    Western Honey Bee   
4              Aquarius remigis  North American Common Water Strider   
5           Argiope trifasciata                 Banded Garden Spider   
6      Belenois creona severina                 African Common White   
7             Buteo jamaicensis                      Red-tailed Hawk   
8                Cathartes aura                       Turkey Vulture   
9           Chromolaena odorata                            Siam weed   
10                 Corvus corax                         Common Raven   
11             Delta fenestrale                                 None   
12     Euthyrhynchus floridanus          Florida Predatory Stink

#### Finding Quarternary Consumers

In [24]:
query = """CALL gds.degree.stream(
  'dense_prey_to_pred',
{ orientation: 'REVERSE' })
YIELD nodeId, score AS indegree
WHERE indegree = 0 
WITH gds.util.asNode(nodeId) AS producer
WHERE producer.taxon_kingdom in ["Plantae", "Chromista"]

// Step 2: Find primary consumers
MATCH (producer)-[:eaten_by]->(primary:Species)

//Step 3: Find Secondary Consumers
MATCH (primary)-[:eaten_by]->(secondary:Species)

//Step 4: Find Tertiary Consumers
MATCH (secondary)-[:eaten_by]->(tertiary:Species)


//step 5: Find Quarternary Consumers
MATCH (tertiary)-[:eaten_by] -> (quaternary:Species)

RETURN DISTINCT quaternary.scientific_name as scientific_name,
        quaternary.common_name as common_name,
        quaternary.taxon_class as class,
        quaternary.taxon_phylum as phylum
ORDER BY quaternary.scientific_name;"""

result = gds.run_cypher(query)
print(result)

                scientific_name                          common_name  \
0                   Afrogegenes                              Dodgers   
1         Andrenosoma hesperium            Golden-horned Chiselmouth   
2        Apiomerus californicus              California Bee Assassin   
3                Apis mellifera                    Western Honey Bee   
4              Aquarius remigis  North American Common Water Strider   
5           Argiope trifasciata                 Banded Garden Spider   
6      Belenois creona severina                 African Common White   
7             Buteo jamaicensis                      Red-tailed Hawk   
8                Cathartes aura                       Turkey Vulture   
9           Chromolaena odorata                            Siam weed   
10             Delta fenestrale                                 None   
11     Euthyrhynchus floridanus          Florida Predatory Stink Bug   
12              Lippia javanica                            Fever

### Herbivores

In [25]:
query = """MATCH (plants:Species)-[:eaten_by]->(herbivore:Species)
WHERE plants.taxon_kingdom IN ['Plantae','Chromista'] and plants.wccId =2 and herbivore.wccId=2
  AND NOT EXISTS {
    MATCH (other:Species)-[:eaten_by]->(herbivore)
    WHERE other.taxon_kingdom IN ['Animalia','Fungi'] and other.wccId=2
  }
RETURN DISTINCT herbivore.scientific_name AS herbivore_name,
       herbivore.common_name AS common_name,
       collect(DISTINCT plants.taxon_kingdom) AS eaten_kingdoms
ORDER BY herbivore_name;"""

result = gds.run_cypher(query)
print(result)

                herbivore_name                  common_name eaten_kingdoms
0              Aceria lantanae      Lantana Flower Gallmite      [Plantae]
1             Aceria theospyri  persimmon leaf blister gall      [Plantae]
2           Adejeania vexatrix                         None      [Plantae]
3          Aerophilus nigripes                         None      [Plantae]
4                  Afrogegenes                      Dodgers      [Plantae]
..                         ...                          ...            ...
210  Zonocerus elegans elegans                         None      [Plantae]
211     Zonotrichia albicollis       White-throated Sparrow      [Plantae]
212           Zosterops virens         Green Cape White-eye      [Plantae]
213    Zosterops virens virens         Green Cape White-eye      [Plantae]
214       Zygaena filipendulae              Six-spot Burnet      [Plantae]

[215 rows x 3 columns]


### Omnivores

In [26]:
query = """MATCH (prey:Species)-[:eaten_by]->(predator:Species)
WHERE prey.wccId =2 AND predator.wccId =2
WITH predator, collect(DISTINCT prey.taxon_kingdom) AS prey_kingdoms
WHERE size(prey_kingdoms) >1
RETURN predator.scientific_name AS omnivore, prey_kingdoms
ORDER BY predator.scientific_name;"""
result = gds.run_cypher(query)
print(result)

                       omnivore               prey_kingdoms
0                Apis mellifera         [Animalia, Plantae]
1                 Canis latrans         [Animalia, Plantae]
2              Coragyps atratus         [Animalia, Plantae]
3           Gallinula chloropus         [Animalia, Plantae]
4            Larus delawarensis         [Animalia, Plantae]
5              Lybius torquatus         [Animalia, Plantae]
6    Lybius torquatus torquatus         [Animalia, Plantae]
7                 Pica hudsonia         [Animalia, Plantae]
8           Pycnonotus barbatus         [Animalia, Plantae]
9   Pycnonotus barbatus layardi         [Animalia, Plantae]
10         Sciurus carolinensis  [Animalia, Plantae, Fungi]
11      Tamiasciurus hudsonicus         [Plantae, Animalia]
12        Tetramorium immigrans         [Animalia, Plantae]
13         Toxomerus marginatus         [Animalia, Plantae]
14      Trachyphonus vaillantii         [Plantae, Animalia]
15           Turdus migratorius         

### Decomposers

In [28]:
query = """MATCH (s:Species)
WHERE s.taxon_kingdom = 'Fungi' AND s.wccId=2
RETURN s.scientific_name AS fungi_species_name;"""
result = gds.run_cypher(query)
print(result)

            fungi_species_name
0              Cercospora atra
1      Coniodictyum chevalieri
2          Coprinopsis lagopus
3         Macrosporium asimini
4     Mycocentrospora asiminae
5     Ophiocordyceps humbertii
6       Phylloporia amplectens
7        Phyllosticta asiminae
8  Pseudocercospora fuliginosa
